# Relevant Segment Extraction (RSE)

Return the best *contiguous run* of chunks instead of isolated top-k chunks.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

Score each chunk, subtract a threshold so irrelevant chunks are negative, then find the maximum-sum contiguous segment (Kadane's algorithm).

In [ ]:
import numpy as np

def best_segments(query, threshold=0.25, max_len=4, n_segments=2):
    scores = store.vecs @ embed(query)[0] - threshold
    segments, used = [], np.zeros(len(scores), dtype=bool)
    for _ in range(n_segments):
        best = (0.0, None)
        for i in range(len(scores)):
            total = 0.0
            for j in range(i, min(i + max_len, len(scores))):
                if used[j]:
                    break
                total += scores[j]
                if total > best[0]:
                    best = (total, (i, j))
        if best[1] is None:
            break
        i, j = best[1]
        used[i:j + 1] = True
        segments.append((best[0], i, j))
    return segments

query = "How do reranking and hybrid search improve retrieval?"
segs = best_segments(query)
for score, i, j in segs:
    print(f"chunks {i}-{j}  score={score:.2f}")

In [ ]:
context = [" ".join(chunks[i:j + 1]) for _, i, j in segs]
print(answer(query, context))